## **Parte IV**
Implementação das tarefas 17 a 20

---


In [1]:
import os
import time

import psycopg2
from psycopg2 import sql, errors
import getpass

In [2]:
## Criando o Database
DB_SUPER = "postgres"
DB_USER = "jaidezardin"
DB_HOST = os.getenv("PGHOST", "localhost")
DB_PORT = int(os.getenv("PGPORT", 5432))

NEW_DB = "reservas"
NEW_TABLE = "Assentos"

conn = psycopg2.connect(dbname=DB_SUPER, user=DB_USER, host=DB_HOST, port=DB_PORT)
conn.autocommit = True
cur = conn.cursor()
try:
    cur.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(NEW_DB)))
    print(f"Banco de dados {NEW_DB} criado.")
except errors.DuplicateDatabase:
    print(f"Banco de dados {NEW_DB} já existe.")
finally:
    cur.close()
    conn.close()

Banco de dados reservas já existe.


In [4]:
## Criando a tabela

conn = psycopg2.connect(dbname=NEW_DB, user=DB_USER, host=DB_HOST, port=DB_PORT)
cur = conn.cursor()

cur.execute(sql.SQL(f"""
CREATE TABLE IF NOT EXISTS {NEW_TABLE} (
num_voo integer,
disp bool,
primary key (num_voo),
check (num_voo > 0),
check (num_voo <= 200)
)
""")
)

conn.commit()
cur.close()
conn.close()


## Tarefa 17
### As duas versões da transação de reserva

A versão A realiza todos os processo em uma única transação, realizando commit apenas no final. Já a versão B realiza a seleção dos assentos disponíveis em uma transação, e depois realiza a atualização em outra transação separada.



In [5]:
## Transações
import random

def reserva_version_a(conn, isolation_level="READ COMMITTED"):
    cur = conn.cursor()
    try:
        # Define o nível da transação
        cur.execute(f"set transaction isolation level {isolation_level}")
        # Seleciona os assentos disponíveis
        cur.execute("select num_voo from Assentos where disp = true")
        open = cur.fetchall()

        # Caso não haja assentos disponíveis, rollback
        if not open:
            conn.rollback()
            return None

        # Dorme por 1 segundo e escolhe aleatoriamente um assento
        time.sleep(1)
        chosen = random.choice(open)[0]

        # Atualiza a tabela de assentos, marcando o assento escolhido como ocupado
        cur.execute(
            """
            update assentos set disp = false where num_voo = %s and disp = true
            """,
            (chosen,)
        )

        # Caso não haja nenhuma linha afetada, rollback
        if cur.rowcount == 0:
            conn.rollback()
            return None

        # Commit da transação
        conn.commit()
        return chosen
    except Exception as e:
        conn.rollback()
        print(f"Erro na reserva versão A: {e}")
        return None
    finally:
        cur.close()


def reserva_version_b(conn, isolation_level="READ COMMITTED"):
    cur = conn.cursor()
    try:
        # Define o nível de isolamento
        cur.execute(f"set transaction isolation level {isolation_level}")

        # Seleciona os assentos disponíveis
        cur.execute("select num_voo from Assentos where disp = true")
        open = cur.fetchall()
        # Commit dessa primeira transação
        conn.commit()

        # Caso não haja nenhum elemento dentro, rollback
        if not open:
            conn.rollback()
            return None

        # Dorme por 1 segundo e escolhe aleatoriamente um assento (sem estar na transação anterior)
        time.sleep(1)
        chosen = random.choice(open)[0]

        # Define o nível de isolamento da próxima transação
        cur.execute(f"set transaction isolation level {isolation_level}")
        # Tenta atualizar a tabela de assentos, marcando o assento escolhido como ocupado
        cur.execute (
            "update Assentos set disp = false where num_voo = %s and disp = true",
            (chosen,)
        )

        # Se atualizou, commit, se não, rollback
        if cur.rowcount == 0:
            conn.rollback()
            return None
        conn.commit()
        return chosen
    except Exception as e:
        conn.rollback()
        print("Erro na reserva versão B: ", e)
        return None
    finally:
        cur.close()


In [6]:
import threading
import time

class MainExecution:
    def __init__(self, reserva_version, k, isolation_level="READ COMMITTED"):
        self.reserva_version = reserva_version
        self.k = k
        self.isolation_level = isolation_level

        self.total_clients = 200
        self.current_k = 0
        self.k_lock = threading.Lock()
        self.counter_lock = threading.Lock()
        self.served_clients = 0


    def setup_database(self):
        # Redefine o banco de dados
        conn = psycopg2.connect(dbname=NEW_DB, user=DB_USER, host=DB_HOST, port=DB_PORT)
        cur = conn.cursor()
        cur.execute(sql.SQL(f"truncate table {NEW_TABLE}"))
        cur.execute(
            f"INSERT INTO {NEW_TABLE} (num_voo, disp) "
            f"SELECT generate_series(1, 200), TRUE"
        )
        conn.commit()
        cur.close()
        conn.close()

    def agent_working(self, agent_id):
        # Cada agente tem sua prórpia conexão com o banco de dados
        conn = psycopg2.connect(dbname=NEW_DB, user=DB_USER, host=DB_HOST, port=DB_PORT)

        # Os agentes executam o processo de reserva de maneira independente, apenas parando quando todos os assentos ja estiverem reservados
        while True:
            with self.counter_lock:
                if self.served_clients >= self.total_clients:
                    break

            result = self.reserva_version(conn, self.isolation_level)

            # Atualização no contador de clientes atendidos
            if result is not None:
                with self.counter_lock:
                    self.served_clients += 1
                    print(f"Agente {agent_id} reservou assento {result}. "
                          f"Total: {self.served_clients}/200")

        # Fecha a conexão
        conn.close()
        print(f"Agente {agent_id} finalizou.")

    def simulate(self):
        print(f"Simulando a execução com k = {self.k} e isolamento {self.isolation_level}")

        self.setup_database()
        self.served_clients = 0
        threads = []
        inicio = time.time()

        # Loop inicializando a quantidade de agentes que estarão em execução
        for i in range(self.k):
            t = threading.Thread(target=self.agent_working, args=(i+1,))
            threads.append(t)
            t.start()

        # Thread principal espera os agentes terminarem
        for t in threads:
            t.join()

        tempo_total = time.time() - inicio
        print(f"Simulação finalizada em {tempo_total:.2f} segundos.")
        print(f"Total de clientes atendidos: {self.served_clients}/200")

        return tempo_total


In [ ]:
## Tarefa 18
### Gráficos de comparação

In [7]:
import matplotlib.pyplot as plt
import pandas as pd

class ExperimentRunner:
    def __init__(self):
        self.results = []

    def run_all_experiments(self):

        versions = [
            ('A', reserva_version_a),
            ('B', reserva_version_b)
        ]

        levels = ['SERIALIZABLE', 'READ COMMITTED']
        k_values = [1, 2, 4, 6, 8, 10]

        for version, reserva_version in versions:
            for level in levels:
                for k in k_values:
                    runner = MainExecution(reserva_version, k, level)
                    self.results.append({
                        'version': version,
                        'isolation_level': level,
                        'k': k,
                        'time': runner.simulate()
                    })



Simulando a execução com k = 5 e isolamento READ COMMITTED
Agente 4 reservou assento 138. Total: 1/200
Agente 1 reservou assento 174. Total: 2/200
Agente 2 reservou assento 52. Total: 3/200
Agente 3 reservou assento 140. Total: 4/200
Agente 5 reservou assento 78. Total: 5/200
Agente 3 reservou assento 151. Total: 6/200
Agente 4 reservou assento 165. Total: 7/200
Agente 1 reservou assento 98. Total: 8/200
Agente 2 reservou assento 83. Total: 9/200
Agente 5 reservou assento 7. Total: 10/200
Agente 3 reservou assento 92. Total: 11/200
Agente 5 reservou assento 112. Total: 12/200
Agente 2 reservou assento 48. Total: 13/200
Agente 1 reservou assento 26. Total: 14/200
Agente 4 reservou assento 114. Total: 15/200
Agente 4 reservou assento 153. Total: 16/200
Agente 1 reservou assento 163. Total: 17/200
Agente 5 reservou assento 108. Total: 18/200
Agente 2 reservou assento 23. Total: 19/200
Agente 3 reservou assento 65. Total: 20/200
Agente 5 reservou assento 193. Total: 21/200
Agente 2 reservo